In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import functions

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import re

In [3]:
df = pd.read_csv("../../data/clean_train.csv", sep=";",encoding="utf-8-sig", keep_default_na=False)

## Create Transcription Headers

In [4]:
df["transcription_headers_model"] = df["transcription"].apply(functions.extract_headers).apply(functions.headers_for_model)
df["transcription_headers_words"] = df["transcription"].apply(functions.extract_headers).apply(functions.headers_words)
df["transcription_headers_words"][0]


'REASON FOR REFERRAL MEDICATIONS ALLERGIES SOCIAL HISTORY REVIEW OF SYSTEMS PEX EKG ECHOCARDIOGRAM (FOR SYNCOPE) IMPRESSION RECOMMENDATIONS'

# TF-IDF

### Train/Test Split

In [5]:
from sklearn.model_selection import train_test_split

train_indices, valid_indices = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["medical_specialty"],
)

y_train = df.loc[train_indices, "medical_specialty"]
y_valid = df.loc[valid_indices, "medical_specialty"]

### Classifiers

In [37]:
linear_svc = LinearSVC(C=0.5,class_weight="balanced",)
logistic_regression = LogisticRegression(C=1.0,class_weight="balanced",max_iter=2000,random_state=42,)
multinomial_nb = MultinomialNB()

### Testing Baseline

In [7]:
input_just_description = functions.build_input_variants(df, ["description"])

pd.DataFrame([functions.evaluate_input_extra(input_just_description, "Just description", multinomial_nb, "Multinomial NB", train_indices, valid_indices, y_train, y_valid)[1]])

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Multinomial NB,Just description,0.384615,0.132751,0.0,Dermatology


### Testing Each Field

In [8]:
input1 = functions.build_input_variants(df, ["description"])
input2 = functions.build_input_variants(df, ["sample_name"])
input3 = functions.build_input_variants(df, ["transcription"])
input4 = functions.build_input_variants(df, ["keywords"])
input5 = functions.build_input_variants(df, ["transcription_headers_model"])
input6 = functions.build_input_variants(df, ["transcription_headers_words"])


#results = pd.DataFrame([
#    functions.evaluate_input_extra(input1, "Description", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input2, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input3, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input4, "keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input5, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input6, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#])

#results

### Testing 2 Fields Combinations

#### Description

In [9]:
input1 = functions.build_input_variants(df, ["description","sample_name"])
input3 = functions.build_input_variants(df, ["description","transcription"])
input4 = functions.build_input_variants(df, ["description","keywords"])
input5 = functions.build_input_variants(df, ["description","transcription_headers_model"])
input6 = functions.build_input_variants(df, ["description","transcription_headers_words"])


#results = pd.DataFrame([
#    functions.evaluate_input_extra(input1, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input3, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input4, "keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input5, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input6, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#])

#results

#### Sample Name

In [10]:
input1 = functions.build_input_variants(df,["description","sample_name"])
input2 = functions.build_input_variants(df, ["sample_name","transcription"])
input3 = functions.build_input_variants(df, ["keywords","sample_name"])
input4 = functions.build_input_variants(df, ["sample_name","transcription_headers_model"])
input5 = functions.build_input_variants(df, ["sample_name","transcription_headers_words"])


#results = pd.DataFrame([
#    functions.evaluate_input_extra(input1, "description", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input2, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input3, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input4, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input5, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#])

#results

#### Keywords

In [11]:
input1 = functions.build_input_variants(df,["description","keywords"])
input2 = functions.build_input_variants(df, ["keywords","transcription"])
input3 = functions.build_input_variants(df, ["keywords","sample_name"])
input4 = functions.build_input_variants(df, ["keywords","transcription_headers_model"])
input5 = functions.build_input_variants(df, ["keywords","transcription_headers_words"])


results = pd.DataFrame([
    functions.evaluate_input_extra(input1, "description", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input2, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input3, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input4, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input5, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
])

results

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear SVC,description,0.831144,0.772146,0.400000,Psychiatry-Psychology
1,Linear SVC,transcription,0.497186,0.534895,0.153846,Neurosurgery
2,Linear SVC,sample name,0.889306,0.864031,0.777778,Psychiatry-Psychology
3,Linear SVC,transcription headers model,0.879925,0.844047,0.705882,Psychiatry-Psychology
4,Linear SVC,transcription headers words,0.883677,0.848029,0.705882,Psychiatry-Psychology


### Keywords and Transcription Headers

Linear SVC 

Logistic Regression

Multinomial Naive Bayes

In [12]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_key = functions.build_input_variants(df, ["keywords"])

#pred1, result1 = functions.evaluate_input_extra(input, "key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred2, result2 = functions.evaluate_input_extra(input_key, "key", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred3, result3 = functions.evaluate_input_extra(input, "key and Headers", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)
#pred4, result4 = functions.evaluate_input_extra(input_key, "key", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)
#pred5, result5 = functions.evaluate_input_extra(input, "key and Headers", multinomial_nb, "multinomial_nb", train_indices, valid_indices, y_train, y_valid)
#pred6, result6 = functions.evaluate_input_extra(input_key, "key", multinomial_nb, "multinomial_nb", train_indices, valid_indices, y_train, y_valid)

#results = pd.DataFrame([result1, result2, result3, result4, result5, result6])
#results

In [13]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_desc = functions.build_input_variants(df, ["description", "keywords", "transcription_headers_model"])
input_desc_sample = functions.build_input_variants(df, ["description", "sample_name", "transcription_headers_model"])
input_sample = functions.build_input_variants(df, ["sample_name", "keywords"])


#pred1, result1 = functions.evaluate_input_extra(input, "key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred2, result2 = functions.evaluate_input_extra(input_desc, "desc, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred3, result3 = functions.evaluate_input_extra(input_desc_sample, "desc, sample, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred4, result4 = functions.evaluate_input_extra(input_sample, "sample, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred5, result5 = functions.evaluate_input_extra(input, "key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#pred6, result6 = functions.evaluate_input_extra(input_desc, "desc, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#pred7, result7 = functions.evaluate_input_extra(input_desc_sample, "desc, sample, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#pred8, result8 = functions.evaluate_input_extra(input_sample, "sample, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#results = pd.DataFrame([result1, result2, result3, result4, result5, result6, result7, result8])
#results

In [14]:
input_keywords = functions.build_input_variants(df, ["description", "keywords", "sample_name", "transcription_headers_model"])
input = functions.build_input_variants(df, ["description", "sample_name", "transcription_headers_model"])
input_only_keywords = functions.build_input_variants(df, ["keywords"])

predict_keywords, result_keywords = functions.evaluate_input_extra(input_keywords, "With Keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
predict, result = functions.evaluate_input_extra(input, "Without Keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
predict_only_keywords, result_only_keywords = functions.evaluate_input_extra(input_only_keywords, "Only Keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
predict_keywords2, result_keywords2 = functions.evaluate_input_extra(input_keywords, "With Keywords", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
predict2, result2 = functions.evaluate_input_extra(input, "Without Keywords", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
predict_only_keywords2, result_only_keywords2 = functions.evaluate_input_extra(input_only_keywords, "Only Keywords", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)

results = pd.DataFrame([result_keywords, result, result_only_keywords, result_keywords2, result2, result_only_keywords2])
results


,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear SVC,With Keywords,0.819887,0.794207,0.651163,Neurosurgery
1,Linear SVC,Without Keywords,0.362101,0.414901,0.102564,Neurosurgery
2,Linear SVC,Only Keywords,0.840525,0.822564,0.166667,Psychiatry-Psychology
3,logistic regression,With Keywords,0.694184,0.692605,0.400000,Neurosurgery
4,logistic regression,Without Keywords,0.452158,0.495792,0.232558,Neurosurgery
5,logistic regression,Only Keywords,0.782364,0.764078,0.166667,Psychiatry-Psychology


In [15]:
labels = sorted(y_valid.unique())
predict_combined = np.where(df.loc[valid_indices, "keywords"] == "", predict2, predict_only_keywords)

functions.show_classification_report(y_valid,predict_combined,labels,)

                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.93      0.89      0.91        62
             Dermatology       1.00      0.80      0.89         5
        Gastroenterology       0.97      0.87      0.92        39
        General Medicine       0.67      0.95      0.78        42
               Neurology       0.78      0.76      0.77        37
            Neurosurgery       0.88      0.88      0.88        16
   Obstetrics-Gynecology       0.96      0.85      0.90        27
           Ophthalmology       0.83      1.00      0.91        15
              Orthopedic       0.90      0.95      0.93        60
   Psychiatry-Psychology       0.75      0.75      0.75         8
               Radiology       0.97      0.89      0.93        44
                 Surgery       0.99      0.95      0.97       178

                accuracy                           0.91       533
               macro avg       0.89      0.88      0.88       533
        

In [16]:
for x in range (0,6):   
    sem_kw = (df.loc[valid_indices, "keywords"] == "").to_numpy()
    pred = [predict_keywords, predict,  predict_only_keywords, predict_keywords2, predict2, predict_only_keywords2][x]
    certo = (pred == y_valid.to_numpy())
    print("com keywords:", certo[~sem_kw].mean(), "| sem keywords:", certo[sem_kw].mean())

com keywords: 0.899103139013453 | sem keywords: 0.41379310344827586
com keywords: 0.34080717488789236 | sem keywords: 0.47126436781609193
com keywords: 0.9887892376681614 | sem keywords: 0.08045977011494253
com keywords: 0.7354260089686099 | sem keywords: 0.4827586206896552
com keywords: 0.4439461883408072 | sem keywords: 0.4942528735632184
com keywords: 0.9192825112107623 | sem keywords: 0.08045977011494253


In [17]:
certo = (predict_combined == y_valid.to_numpy())
errado = (predict_combined != y_valid.to_numpy())
print("com keywords:", certo[~sem_kw].mean(), "| sem keywords:", certo[sem_kw].mean())

print(certo.sum(), certo[~sem_kw].sum(), certo[sem_kw].sum())

print(errado.sum(), errado[~sem_kw].sum(), errado[sem_kw].sum())

com keywords: 0.9887892376681614 | sem keywords: 0.4942528735632184
484 441 43
49 5 44


In [18]:
indices_erros_com_kw = valid_indices[errado & (~sem_kw)]
indices_erros_sem_kw = valid_indices[errado & sem_kw]
indices_certos = valid_indices[certo]

print(f"Número de erros COM keywords: {len(indices_erros_com_kw)}")
print(f"Número de erros SEM keywords: {len(indices_erros_sem_kw)}")
print(f"Número de acertos: {len(indices_certos)}")

Número de erros COM keywords: 5
Número de erros SEM keywords: 44
Número de acertos: 484


In [22]:
df_certos = pd.DataFrame({
    'Real': y_valid.loc[indices_certos],
    'Previsto': predict_combined[certo], 
    'Description': df.loc[indices_certos, 'description'],
    'Keywords': df.loc[indices_certos, 'keywords']
})


df_certos.head(10)

,Real,Previsto,Description,Keywords
2305,Surgery,Surgery,Colonoscopy. Change in bowel habits and recta...,"surgery, olympus, colonoscope, bowel habits, c..."
2015,Surgery,Surgery,Cataract extraction with phacoemulsification a...,"surgery, provisc, intraocular lens implantatio..."
1989,Surgery,Surgery,Right orchiopexy and right inguinal hernia rep...,"surgery, ectopic position, inguinal hernia, in..."
973,Neurosurgery,Neurosurgery,"Revision laminectomy L5-S1, discectomy L5-S1, ...","neurosurgery, revision laminectomy, discectomy..."
2616,Surgery,Surgery,"Cataract, right eye. Phacoemulsification with...","surgery, tetracaine, intraocular lens, lid spe..."
1425,Orthopedic,Orthopedic,Incision and drainage and removal of foreign b...,"orthopedic, incision and drainage, removal of ..."
54,Cardiovascular-Pulmonary,Cardiovascular-Pulmonary,Pulmonary function test. Mild-to-moderate obs...,"cardiovascular / pulmonary, lung volume, force..."
527,General Medicine,General Medicine,Patient had a piece of glass fall on to his ri...,"general medicine, foot pain, cut on foot, piec..."
2008,Surgery,Surgery,"Left orchiectomy, scrotal exploration, right o...","surgery, testicular abscess, necrotic testes, ..."
1839,Surgery,Surgery,Screening colonoscopy. Tiny polyps. If aden...,"surgery, pentax video colonoscope, biopsy forc..."


In [ ]:
# Criar uma tabela com os erros do grupo SEM keywords
df_erros_sem_kw = pd.DataFrame({
    'Real': y_valid.loc[indices_erros_sem_kw],
    'Previsto': predict_combined[errado & sem_kw], 
    'Description': df.loc[indices_erros_sem_kw, 'description'],
    'Keywords': df.loc[indices_erros_sem_kw, 'keywords']
})


df_erros_sem_kw


,Real,Previsto,Description,Keywords
207,Cardiovascular-Pulmonary,General Medicine,Aspiration pneumonia and chronic obstructive p...,
1725,Radiology,Neurology,MRI Brain & MRI C-T spine: Multiple hemangiobl...,
1,Cardiovascular-Pulmonary,General Medicine,"The patient is 14 months old, comes in with a ...",
1503,Psychiatry-Psychology,General Medicine,Falls at home. Anxiety and depression. The p...,
967,Neurosurgery,Neurology,Right side craniotomy for temporal lobe intrac...,
687,General Medicine,Neurology,Followup after a full-night sleep study perfor...,
369,Gastroenterology,General Medicine,"Status post colonoscopy. After discharge, exp...",
2447,Surgery,Cardiovascular-Pulmonary,"Left heart catheterization, bilateral selectiv...",
1768,Surgery,Obstetrics-Gynecology,"Dilation and curettage (D&C), laparoscopy, and...",
2340,Cardiovascular-Pulmonary,General Medicine,"Elevated BNP. Diastolic heart failure, not co...",


In [23]:
df_erros_com_kw = pd.DataFrame({
    'Real': y_valid.loc[indices_erros_com_kw],
    'Previsto': predict_combined[errado & ~sem_kw], 
    'Keywords': df.loc[indices_erros_com_kw, 'keywords']
})

df_erros_com_kw

,Real,Previsto,Keywords
2580,Surgery,Neurosurgery,"surgery, metastatic lesion, biparietal, mayfie..."
2635,Surgery,Ophthalmology,"surgery, ruptured globe, ancef, bss, maxitrol ..."
818,Neurology,Radiology,"neurology, emg, nerve conduction study, nerve ..."
881,Neurology,Orthopedic,"neurology, exiting c nerve roots, loss of disc..."
1941,Surgery,Ophthalmology,"surgery, lid speculum, conjunctival, peritomy,..."


In [34]:
verificacao = pd.DataFrame({
    "Indice": valid_indices,
    "Real": y_valid.to_numpy(),
    "Previsto": predict_combined,
    "Sem keywords": sem_kw,
    "Errado": errado
})

previsao_general_medicine = (
    predict_combined == "General Medicine"
)

indices_previstos_general_medicine = (
    valid_indices[previsao_general_medicine & errado & sem_kw]
)

df.loc[indices_previstos_general_medicine, "medical_specialty"].value_counts()

medical_specialty
Cardiovascular-Pulmonary    6
Gastroenterology            5
Neurology                   4
Obstetrics-Gynecology       2
Psychiatry-Psychology       1
Dermatology                 1
Orthopedic                  1
Name: count, dtype: int64

In [35]:
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# o evaluate_input_extra não devolve o modelo, por isso refaz-se (deve reproduzir o predict2)
model2 = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50_000)),
    ("classifier", clone(logistic_regression)),
]).fit(input.loc[train_indices], y_train)        # input = description + sample_name + headers

proba = model2.predict_proba(input.loc[valid_indices])
classes = list(model2.classes_)

for idx in [13, 2660, 2088, 1503, 766]:
    i = valid_indices.get_loc(idx)
    ordem = list(np.array(classes)[np.argsort(-proba[i])])
    real = y_valid.loc[idx]
    print(f"{idx}: real={real} (posição {ordem.index(real)+1}, p={proba[i, classes.index(real)]:.2f})"
          f" | previsto={ordem[0]} (p={proba[i].max():.2f})")
    print("   sample_name:", df.loc[idx, "sample_name"])
    print("   description:", df.loc[idx, "description"][:200])
    print("   headers:", df.loc[idx, "transcription_headers_words"][:100])

13: real=Cardiovascular-Pulmonary (posição 2, p=0.31) | previsto=General Medicine (p=0.31)
   sample_name: Consult - Congestive Heart Failure
   description: Congestive heart failure (CHF).  The patient is a 75-year-old gentleman presented through the emergency room.  Symptoms are of shortness of breath, fatigue, and tiredness.  Main complaints are right-s
   headers: REASON FOR CONSULTATION HISTORY OF PRESENT ILLNESS CORONARY RISK FACTORS FAMILY HISTORY PAST SURGICA
2660: real=Surgery (posição 2, p=0.13) | previsto=Orthopedic (p=0.63)
   sample_name: Arthroscopy Shoulder/Knee
   description: Right shoulder arthroscopy, subacromial decompression, distal clavicle excision, bursectomy, and coracoacromial ligament resection, carpal tunnel release, left knee arthroscopy, and partial medial and
   headers: PREOPERATIVE DIAGNOSES POSTOPERATIVE DIAGNOSES PROCEDURE ANESTHESIA COMPLICATIONS DISPOSITION OPERAT
2088: real=Cardiovascular-Pulmonary (posição 6, p=0.05) | previsto=Surgery (p=0.23)
  